# InterventionROP-Edge: Complete Reproducible Notebook

**Paper:** InterventionROP-Edge: Do-Calculus-Guided Threshold-Aware Compute-Bandwidth Orchestration for Retinopathy of Prematurity Triage in Neonatal Edge-IoT Systems

**Run:** `Runtime > Run all`. CPU runtime is enough. Set `QUICK = True` in the configuration cell for a short smoke test (about 5 minutes).

**What is real and what is simulated**
1. Real: gestational age, birth weight, postconceptual age, ICROP diagnosis, plus form, device and the longitudinal series of the Ostrava cohort (Timkovič et al., Sci. Data 11, 814, 2024, DOI 10.1038/s41597-024-03409-7). The clinical table is downloaded directly from the authors' GitHub repository.
2. Semi synthetic: oxygen exposure, treatment assignment and the two potential outcomes are generated by the structural causal model calibrated to the real cohort (the only way to know the true counterfactual effect).
3. Simulated: the 20 node NICU IEEE 802.15.4 edge network.

**Corrections relative to the earlier package**
1. No ground truth leakage: the controller's digital twin estimates arrival and crossing rates online from observations only (the earlier twin read `env.arrival_p`, which is computed from the true urgency).
2. Fair comparison: in the main experiment every method receives the same causal urgency input; the value of causal urgency is tested separately.
3. The do-calculus constraint is a real interventional constraint: candidate actions are evaluated by the twin under do(action), and actions whose predicted latency for threshold risk infants violates the deadline are rejected.
4. No pseudo replication: the semi synthetic cohort uses the real 457 series (no resampling to 3000 rows by default) and confidence intervals use a patient level cluster bootstrap.
5. The correlational and unstratified estimators use the same learner as the causal one; only the confounder adjustment or the stratification differs.
6. Every percentage in the results table is computed from the run, with its sign. Report exactly what this notebook prints.

## 1. Setup

In [ ]:
import sys, subprocess, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run("pip -q install networkx openpyxl", shell=True)
    subprocess.run("echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true | debconf-set-selections; "
                   "apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y ttf-mscorefonts-installer > /dev/null 2>&1; "
                   "rm -rf ~/.cache/matplotlib", shell=True)

import io, json, time, copy, math, shutil, warnings
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.special import comb
from scipy.stats import wilcoxon, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

warnings.filterwarnings("ignore")
torch.set_num_threads(2)
print("torch", torch.__version__, "| pandas", pd.__version__)

## 2. Configuration

In [ ]:
QUICK = False                      # True = short smoke test

SEED = 0
GAMMA_DEFAULT = 1.5                # oxygen confounding strength of the default scenario
TRAIN_EPISODES = 6 if QUICK else 100
TRAIN_ROUNDS = 200
EVAL_ROUNDS = 120 if QUICK else 400
EVAL_SEEDS = [5000 + s for s in range(2 if QUICK else 10)]
SWEEP_SEEDS = EVAL_SEEDS[:2 if QUICK else 5]
NODE_LIST = [10, 20, 30, 40]
NOISE_LIST = [-6, -3, 0, 3, 6]
LIFE_SEEDS = [7000, 7001, 7002]
LIFE_MAX_ROUNDS = 400 if QUICK else 6000
LIFE_BATTERY_J = 300.0
B_BOOT = 30 if QUICK else 200      # patient level cluster bootstrap replicates
MC_REPS = 3 if QUICK else 20       # Monte Carlo replicates per confounding level
GAMMAS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
N_AMP = 0                          # 0 = use the real series only (recommended)

DATA_URL = "https://raw.githubusercontent.com/JanaNowakova/Retinal-Image-Dataset-of-Infants-and-ROP/main/infant_retinal_database_info.csv"
XLSX_URL = "https://raw.githubusercontent.com/JanaNowakova/Retinal-Image-Dataset-of-Infants-and-ROP/main/infant_retinal_database_info.xlsx"

PAPER = "InterventionROP-Edge"
OUT_DIR = (Path("/content/Downloads") if IN_COLAB else Path.home() / "Downloads") / PAPER
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

METHODS = ["InterventionROP-Edge", "HGAT-PPO", "PPO-RA", "PD-NoisyDQN", "DT-QMIX", "JOFO-FedDQN"]
BASELINES = METHODS[1:]
ABLATIONS = ["Abl-NoDoConstraint", "Abl-NoTwin", "Abl-NoSurrogate", "Abl-CorrUrgency", "Abl-Unstratified"]

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

RESULTS = {"config": {k: v for k, v in dict(QUICK=QUICK, SEED=SEED, GAMMA_DEFAULT=GAMMA_DEFAULT, TRAIN_EPISODES=TRAIN_EPISODES,
           TRAIN_ROUNDS=TRAIN_ROUNDS, EVAL_ROUNDS=EVAL_ROUNDS, EVAL_SEEDS=EVAL_SEEDS, SWEEP_SEEDS=SWEEP_SEEDS,
           B_BOOT=B_BOOT, MC_REPS=MC_REPS, N_AMP=N_AMP).items()}}
print("Output folder:", OUT_DIR)

## 3. Dataset (direct download)

In [ ]:
raw = pd.read_csv(DATA_URL, sep=";", encoding="utf-8-sig")
print(raw.shape)
print(raw.head().to_string())

try:
    dg_list = pd.read_excel(XLSX_URL, sheet_name="diagnosis").dropna(how="all")
    print(dg_list.to_string(index=False))
except Exception as e:
    print("Diagnosis list not loaded:", e)

# Official coding (xlsx sheet 'diagnosis'): 0 physiological, 1 ROP 0, 2 ROP 1, 3 ROP 2, 4 ROP 3, 5 ROP 4A, 6 ROP 4B,
# 7 ROP 5, 8 AP-ROP, 9 status post ROP, 10 hamartoma, 11 hemorrhage, 12 hypoplasia n.II, 13 toxoplasma.
DG_STAGE = {0: 0, 1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 4, 7: 5, 8: 3, 9: -1, 10: 0, 11: 0, 12: 0, 13: 0}
ROP_CODES = {1, 2, 3, 4, 5, 6, 7, 8, 9}
GA_BINS = [0, 28, 32, 36, 99]
GA_LABELS = ["GA<=28", "GA 29-32", "GA 33-36", "GA>=37"]


def prepare(raw):
    df = raw.copy()
    df.columns = ["id", "sex", "ga", "bw", "pa", "dg", "pf", "dev", "s"]
    df["sex"] = (df.sex == "M").astype(int)
    df["is_rop"] = df.dg.isin(ROP_CODES).astype(int)
    df["stage"] = df.dg.map(DG_STAGE)
    df["aprop"] = (df.dg == 8).astype(int)
    df["plus"] = (df.pf == 2).astype(int)
    df["treat_indicated"] = ((df.plus == 1) | (df.aprop == 1) | (df.stage >= 3)).astype(int)
    df["ga_stratum"] = pd.cut(df.ga, GA_BINS, labels=GA_LABELS).astype(str)
    df = df.sort_values(["id", "pa", "s"]).reset_index(drop=True)
    g = df.groupby("id")
    df["stage_next"] = g.stage.shift(-1)
    df["dg_next"] = g.dg.shift(-1)
    df["pa_next"] = g.pa.shift(-1)
    nxt = df.stage_next.notna()
    df["regress_next"] = np.where(nxt, ((df.stage_next < df.stage) | (df.dg_next == 9)).astype(float), np.nan)
    df["progress_next"] = np.where(nxt, (df.stage_next > df.stage).astype(float), np.nan)
    return df


df = prepare(raw)
rop_fu = df[(df.is_rop == 1) & df.stage_next.notna() & (df.stage >= 0)]
DATASET = {
    "n_patients": int(df.id.nunique()), "n_series": int(len(df)),
    "n_rop_patients": int(df[df.is_rop == 1].id.nunique()), "n_rop_series": int(df.is_rop.sum()),
    "n_rop_series_with_followup": int(len(rop_fu)),
    "regression_rate": float(rop_fu.regress_next.mean()), "progression_rate": float(rop_fu.progress_next.mean()),
    "mean_followup_weeks": float((rop_fu.pa_next - rop_fu.pa).mean()),
    "treat_indicated_patients": int(df[df.treat_indicated == 1].id.nunique()),
    "treat_indicated_series": int(df.treat_indicated.sum()),
    "dg_counts": {int(k): int(v) for k, v in df.dg.value_counts().sort_index().items()},
    "patients_per_ga_stratum": {k: int(v) for k, v in df.groupby("id").ga_stratum.first().value_counts().items()},
}
RESULTS["dataset"] = DATASET
print(json.dumps(DATASET, indent=1))

## 4. Structural causal model and counterfactual threshold estimator

In [ ]:
TREAT, OUTCOME = "T", "Reg"


def build_dag():
    G = nx.DiGraph()
    G.add_edges_from([("GA", "BW"), ("GA", "O2"), ("BW", "O2"), ("GA", "Stage"), ("BW", "Stage"), ("O2", "Stage"),
                      ("PA", "Stage"), ("Stage", TREAT), ("GA", TREAT), ("O2", TREAT), (TREAT, OUTCOME),
                      ("Stage", OUTCOME), ("O2", OUTCOME), ("PA", OUTCOME), ("GA", OUTCOME), ("BW", OUTCOME), ("Sex", "BW")])
    return G


def d_sep(G, x, y, z):
    if hasattr(nx, "is_d_separator"):
        return nx.is_d_separator(G, {x}, {y}, set(z))
    return nx.d_separated(G, {x}, {y}, set(z))


def minimal_backdoor_sets(G, t=TREAT, y=OUTCOME):
    desc = nx.descendants(G, t)
    cand = [v for v in G.nodes if v not in desc and v not in (t, y)]
    Gt = G.copy()
    Gt.remove_edges_from(list(G.out_edges(t)))
    for k in range(len(cand) + 1):
        valid = [tuple(sorted(Z)) for Z in combinations(cand, k) if d_sep(Gt, t, y, Z)]
        if valid:
            return valid
    return []


DAG = build_dag()
RESULTS["backdoor_sets"] = [list(z) for z in minimal_backdoor_sets(DAG)]
print("Minimal back door adjustment sets:", RESULTS["backdoor_sets"])


def sig(x):
    return 1.0 / (1.0 + np.exp(-x))


def calibrate_intercept(logit, target, lo=-12.0, hi=12.0):
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if sig(logit + mid).mean() > target:
            hi = mid
        else:
            lo = mid
    return 0.5 * (lo + hi)


def generate_semisynthetic(df, gamma=GAMMA_DEFAULT, seed=0, n_amp=N_AMP, reg_rate=None):
    """Real covariates; O2, T, Y(0), Y(1) generated from the SCM calibrated to the real cohort."""
    rng = np.random.default_rng(seed)
    d = df[df.stage >= 0].copy().reset_index(drop=True)
    if n_amp and n_amp > len(d):
        d = d.iloc[rng.integers(0, len(d), n_amp)].reset_index(drop=True)
    n = len(d)
    ga, bw, pa = d.ga.values.astype(float), d.bw.values.astype(float), d.pa.values.astype(float)
    stage, plus, aprop = d.stage.values.astype(float), d.plus.values.astype(float), d.aprop.values.astype(float)
    sev = np.clip(stage, 0, 3) / 3.0 + 0.5 * plus + 0.5 * aprop
    o2 = sig(0.55 * (30.0 - ga) + 0.8 * (1500.0 - bw) / 500.0 + rng.normal(0, 0.8, n))
    thr = ((stage >= 3) | (plus == 1) | (aprop == 1)).astype(float)
    lt = 2.5 * thr + 0.9 * sev + gamma * 1.5 * (o2 - 0.5) + 0.3 * (28.0 - ga) / 4.0
    lt = lt + calibrate_intercept(lt, max(float(d.treat_indicated.mean()), 0.05))
    pT = sig(lt)
    T = (rng.uniform(size=n) < pT).astype(float)
    tau = 0.3 + 2.8 * thr + 0.6 * thr * np.clip(1.0 - np.abs(ga - 29.0) / 3.0, -1, 1) - 0.8 * (o2 - 0.5) * thr
    base = (0.6 * (pa - 36.0) / 4.0 - gamma * 1.6 * (o2 - 0.5) - 1.4 * thr - 0.5 * (stage == 2)
            + 0.25 * (ga - 30.0) / 4.0 + 0.2 * (bw - 1700.0) / 700.0)
    reg_rate = DATASET["regression_rate"] if reg_rate is None else reg_rate
    rop = d.is_rop.values.astype(bool)
    a0 = calibrate_intercept((base + tau * pT)[rop], reg_rate)
    l0 = a0 + base + 2.5 * (~rop)
    l1 = l0 + tau
    p0, p1 = sig(l0), sig(l1)
    u = rng.uniform(size=n)
    Y0, Y1 = (u < p0).astype(float), (u < p1).astype(float)
    return pd.DataFrame({"id": d.id.values, "ga": ga, "bw": bw, "pa": pa, "stage": stage, "plus": plus, "aprop": aprop,
                         "is_rop": d.is_rop.values, "O2": o2, "T": T, "Y": np.where(T == 1, Y1, Y0), "Y0": Y0, "Y1": Y1,
                         "p0": p0, "p1": p1, "ite": p1 - p0, "ga_stratum": d.ga_stratum.values})


BASE_FEATS = ["ga", "bw", "pa", "stage", "plus", "aprop"]
# name: (adjust for the O2 confounder, gestational age stratification)
ESTIMATORS = {"causal": (True, True), "corr": (False, True), "unstrat": (True, False)}
EST_LABEL = {"causal": "Stratified do-calculus (proposed)", "corr": "Correlational (no O2 adjustment)",
             "unstrat": "Unstratified adjustment"}


def design(d, adjust_o2, stratified):
    thr = ((d.stage >= 3) | (d.plus == 1) | (d.aprop == 1)).values.astype(float)[:, None]
    cols = [d[BASE_FEATS].values, thr, thr * d.ga.values[:, None]]
    if adjust_o2:
        cols += [d.O2.values[:, None], thr * d.O2.values[:, None]]
    if stratified:
        oh = np.column_stack([(d.ga_stratum == g).values.astype(float) for g in GA_LABELS])
        cols += [oh, oh * thr]
    return np.column_stack(cols)


class Const:
    def __init__(self, v):
        self.v = v

    def predict_proba(self, X):
        return np.column_stack([1 - np.full(len(X), self.v), np.full(len(X), self.v)])


def fit_lr(X, y):
    if len(y) < 6 or len(np.unique(y)) < 2:
        return Const(float(np.clip(np.mean(y) if len(y) else 0.2, 0.02, 0.98)))
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000)).fit(X, y)


def estimate(d, kind):
    """T learner under the back door adjustment implied by `kind`; returns (p0_hat, p1_hat)."""
    X = design(d, *ESTIMATORS[kind])
    T = d["T"].values == 1
    m1, m0 = fit_lr(X[T], d.Y.values[T]), fit_lr(X[~T], d.Y.values[~T])
    return m0.predict_proba(X)[:, 1], m1.predict_proba(X)[:, 1]


def aipw(d, kind, p0, p1, mask, clip=0.02):
    X = design(d, *ESTIMATORS[kind])[mask]
    T, Y = d["T"].values[mask], d.Y.values[mask]
    if len(np.unique(T)) < 2:
        return float("nan")
    ps = np.clip(fit_lr(X, T).predict_proba(X)[:, 1], clip, 1 - clip)
    dr = (p1[mask] - p0[mask]) + T * (Y - p1[mask]) / ps - (1 - T) * (Y - p0[mask]) / (1 - ps)
    return float(dr.mean())


def attribution(d, p0h, p1h, thr=0.5):
    """Among treated regressions: was regression caused by treatment (Y1=1, Y0=0)?"""
    reg = (d.Y.values == 1) & (d["T"].values == 1)
    truth = (d.Y1.values == 1) & (d.Y0.values == 0)
    score = 1.0 - np.clip(p0h, 1e-6, 1) / np.clip(p1h, 1e-6, 1)
    pred = score > thr
    tp, fp, fn = np.sum(pred & truth & reg), np.sum(pred & ~truth & reg), np.sum(~pred & truth & reg)
    auc = float(roc_auc_score(truth[reg], score[reg])) if len(np.unique(truth[reg])) == 2 else float("nan")
    return {"precision": float(tp / max(tp + fp, 1)), "recall": float(tp / max(tp + fn, 1)), "auroc": auc, "n_events": int(reg.sum())}


def pehe(d, p0h, p1h):
    return float(np.sqrt(np.mean(((p1h - p0h) - d.ite.values) ** 2)))


def urgency(p0h, p1h, stage, plus, aprop):
    """Counterfactual threshold urgency in [0.02, 1]: ICROP severity x risk without treatment x treatment benefit."""
    sev = np.where((stage >= 3) | (plus == 1) | (aprop == 1), 1.0, np.where(stage == 2, 0.7, np.where(stage == 1, 0.45, 0.25)))
    tau = np.clip((np.clip(p1h, 0, 1) - np.clip(p0h, 0, 1)) / 0.6, 0, 1)
    return np.clip(sev * (1.0 - np.clip(p0h, 0, 1)) * (0.3 + 0.7 * tau), 0.02, 1.0)


def groups_of(d):
    rop = d.is_rop.values == 1
    return {"GA<=28, all series": (d.ga_stratum == "GA<=28").values,
            "GA 29-32, all series": (d.ga_stratum == "GA 29-32").values,
            "ROP series, GA<=28": rop & (d.ga_stratum == "GA<=28").values,
            "All ROP series": rop,
            "Whole screening cohort": np.ones(len(d), bool)}


def cluster_bootstrap_index(ids, rng):
    uniq = np.unique(ids)
    rows = {p: np.where(ids == p)[0] for p in uniq}
    pick = rng.choice(uniq, len(uniq), replace=True)
    return np.concatenate([rows[p] for p in pick])


def ci(v):
    v = np.asarray([x for x in v if np.isfinite(x)])
    if len(v) == 0:
        return {"mean": float("nan"), "lo": float("nan"), "hi": float("nan")}
    return {"mean": float(v.mean()), "lo": float(np.percentile(v, 2.5)), "hi": float(np.percentile(v, 97.5))}

In [ ]:
log("causal experiment, default scenario")
data = generate_semisynthetic(df, gamma=GAMMA_DEFAULT, seed=SEED)
C = {"n_series": int(len(data)), "n_patients": int(data.id.nunique()), "n_treated": int(data["T"].sum()),
     "treat_rate": float(data["T"].mean()), "reg_rate": float(data.Y.mean()), "true_ate": float(data.ite.mean()),
     "n_treated_by_stratum": {g: int(((data.ga_stratum == g) & (data["T"] == 1)).sum()) for g in GA_LABELS}}

fits = {k: estimate(data, k) for k in ESTIMATORS}
for k, (p0h, p1h) in fits.items():
    C[f"pehe_{k}"] = pehe(data, p0h, p1h)
    C[f"attribution_{k}"] = attribution(data, p0h, p1h)
C["attribution_oracle"] = attribution(data, data.p0.values, data.p1.values)

rng = np.random.default_rng(SEED + 1)
grp_true = {g: float(data.ite.values[m].mean()) for g, m in groups_of(data).items()}
boot = {g: {k: [] for k in list(ESTIMATORS) + ["aipw"]} for g in grp_true}
for b in range(B_BOOT):
    bd = data.iloc[cluster_bootstrap_index(data.id.values, rng)].reset_index(drop=True)
    bf = {k: estimate(bd, k) for k in ESTIMATORS}
    for g, m in groups_of(bd).items():
        if m.sum() == 0:
            continue
        for k, (q0, q1) in bf.items():
            boot[g][k].append(float((q1 - q0)[m].mean()))
        boot[g]["aipw"].append(aipw(bd, "causal", *bf["causal"], m))
C["ate_groups"] = {}
for g, m in groups_of(data).items():
    row = {"true": grp_true[g], "n": int(m.sum()), "n_treated": int(data["T"].values[m].sum())}
    for k, v in boot[g].items():
        row[k] = ci(v)
        row[k]["covers_truth"] = bool(row[k]["lo"] <= grp_true[g] <= row[k]["hi"])
    C["ate_groups"][g] = row

p0c, p1c = fits["causal"]
C["ite_by_stage"] = {str(s): {"est": [float(x) for x in (p1c - p0c)[data.stage.values == s]],
                              "true": [float(x) for x in data.ite.values[data.stage.values == s]]}
                     for s in [0, 1, 2, 3] if (data.stage.values == s).sum() > 3}
RESULTS["causal"] = C
print(pd.DataFrame({g: {"true": v["true"], "n_treated": v["n_treated"],
                        **{k: f"{v[k]['mean']:.3f} [{v[k]['lo']:.3f}, {v[k]['hi']:.3f}]" for k in ["causal", "corr", "unstrat", "aipw"]}}
                    for g, v in C["ate_groups"].items()}).T.to_string())
print({k: round(C[f"pehe_{k}"], 4) for k in ESTIMATORS})
print({k: C[f"attribution_{k}"] for k in list(ESTIMATORS) + ["oracle"]})

Note: the real cohort has only 5 treatment indicated infants, so the semi synthetic data contain few treated series and treated and untreated series barely overlap outside the threshold group. Effects for the whole cohort and for GA 29 to 32 are therefore extrapolations; read the `covers_truth` flags and report the intervals as they are.

In [ ]:
log("confounding sweep")
sweep = {"gamma": GAMMAS}
keys = ["pehe", "ate_err", "auroc", "precision", "urank"]
for k in keys:
    for e in ESTIMATORS:
        sweep[f"{k}_{e}"] = []
        sweep[f"{k}_{e}_sd"] = []
sweep["auroc_oracle"] = []
for gm in GAMMAS:
    acc = {f"{k}_{e}": [] for k in keys for e in ESTIMATORS}
    orc = []
    for r in range(MC_REPS):
        dd = generate_semisynthetic(df, gamma=gm, seed=SEED + 1000 * r + int(gm * 10))
        ut = urgency(dd.p0.values, dd.p1.values, dd.stage.values, dd.plus.values, dd.aprop.values)
        for e in ESTIMATORS:
            q0, q1 = estimate(dd, e)
            at = attribution(dd, q0, q1)
            acc[f"pehe_{e}"].append(pehe(dd, q0, q1))
            acc[f"ate_err_{e}"].append(abs(float((q1 - q0).mean()) - float(dd.ite.mean())))
            acc[f"auroc_{e}"].append(at["auroc"])
            acc[f"precision_{e}"].append(at["precision"])
            acc[f"urank_{e}"].append(float(spearmanr(ut, urgency(q0, q1, dd.stage.values, dd.plus.values, dd.aprop.values))[0]))
        orc.append(attribution(dd, dd.p0.values, dd.p1.values)["auroc"])
    for k, v in acc.items():
        sweep[k].append(float(np.nanmean(v)))
        sweep[k + "_sd"].append(float(np.nanstd(v)))
    sweep["auroc_oracle"].append(float(np.nanmean(orc)))
RESULTS["confounding_sweep"] = sweep
print(pd.DataFrame({k: v for k, v in sweep.items() if not k.endswith("_sd")}).round(3).to_string(index=False))

## 5. Urgency signals for the NICU cohort

In [ ]:
def build_cohort(df, seed=SEED):
    d = generate_semisynthetic(df, gamma=GAMMA_DEFAULT, seed=seed)
    st, pl, ap = d.stage.values, d.plus.values, d.aprop.values
    d["u_true"] = urgency(d.p0.values, d.p1.values, st, pl, ap)
    for k in ESTIMATORS:
        q0, q1 = estimate(d, k)
        d[f"u_{k}"] = urgency(q0, q1, st, pl, ap)
    return d


cohort = build_cohort(df)
RESULTS["cohort"] = {"n": int(len(cohort)), "u_true_mean": float(cohort.u_true.mean()),
                     **{f"spearman_u_{k}": float(spearmanr(cohort.u_true, cohort[f"u_{k}"])[0]) for k in ESTIMATORS},
                     **{f"mae_u_{k}": float(np.abs(cohort.u_true - cohort[f"u_{k}"]).mean()) for k in ESTIMATORS}}
print(RESULTS["cohort"])

## 6. NICU IEEE 802.15.4 edge-IoT simulator

In [ ]:
ROUND_SEC = 60.0
RATE_BPS = 250e3
FRAME_BITS = 127 * 8
TX_POWER_DBM, NOISE_DBM = -10.0, -76.0
PL0_DB, PL_EXP, SHADOW_SIGMA = 45.0, 3.3, 4.0
E_TX_J_PER_BIT, E_RX_J_PER_BIT = 0.0524 / RATE_BPS, 0.0564 / RATE_BPS
E_IDLE_ROUND = 0.012
E_LOCAL_J_PER_CYCLE = 0.6e-9
F_LOCAL, F_GW, F_EDGE = 0.1e9, 2.0e9, 24.0e9
IMG_BITS_RANGE = (200e3 * 8, 600e3 * 8)
E_CAPTURE_J = 0.15
CAPTURE_SCALE = np.array([0.5, 1.0, 1.6])
TELEMETRY_BITS = 480 * 8
INFER_CYCLES = 4.0e9
RESULT_BITS = 1e3 * 8
DEADLINE_SEC = 180.0
CSMA_EFF = 0.85


def ber_oqpsk(snr_lin):
    k = np.arange(2, 17)
    s = np.atleast_1d(snr_lin)[:, None]
    val = (8.0 / 15.0) * (1.0 / 16.0) * np.sum(((-1.0) ** k) * comb(16, k) * np.exp(20.0 * s * (1.0 / k - 1.0)), axis=1)
    return np.clip(val, 1e-9, 0.5)


def per_from_snr_db(snr_db, bits=FRAME_BITS):
    return 1.0 - (1.0 - ber_oqpsk(10 ** (np.asarray(snr_db) / 10.0))) ** bits


class NICUEnv:
    def __init__(self, n_nodes=20, n_gateways=3, seed=0, battery_j=600.0, interference_prob=0.10,
                 noise_offset_db=0.0, cohort=None, shift=None, max_rounds=400):
        self.rng = np.random.default_rng(seed)
        self.N, self.K, self.B = n_nodes, n_gateways, battery_j
        self.p_int, self.noise_offset = interference_prob, noise_offset_db
        self.cohort, self.shift, self.max_rounds = cohort, shift, max_rounds
        self._place()

    def _place(self):
        room = (16.0, 10.0)
        cols = int(np.ceil(np.sqrt(self.N * room[0] / room[1])))
        rows = int(np.ceil(self.N / cols))
        pts = np.array([(x, y) for y in np.linspace(1.5, room[1] - 1.5, rows) for x in np.linspace(1.5, room[0] - 1.5, cols)])[: self.N]
        self.pos = pts + self.rng.normal(0, 0.3, pts.shape)
        gx = np.linspace(room[0] / (2 * self.K), room[0] - room[0] / (2 * self.K), self.K)
        self.gpos = np.column_stack([gx, np.full(self.K, room[1] / 2.0)])
        d = np.linalg.norm(self.pos[:, None, :] - self.gpos[None, :, :], axis=2)
        self.gw = np.argmin(d, axis=1)
        self.dist = np.maximum(d[np.arange(self.N), self.gw], 1.0)

    def _sample_infants(self):
        c = self.cohort
        w = np.where(c.ga.values <= 28, 4.0, np.where(c.ga.values <= 32, 1.5, 0.3)) if self.shift == "ga_shift" else np.ones(len(c))
        self.infant = c.iloc[self.rng.choice(len(c), self.N, replace=True, p=w / w.sum())].reset_index(drop=True)
        self.u_true = self.infant.u_true.values.astype(float)            # ground truth, used only by the simulator and metrics
        for k in ESTIMATORS:                                              # estimated urgency, observable by controllers
            setattr(self, f"u_{k}", np.clip(self.infant[f"u_{k}"].values + self.rng.normal(0, 0.02, self.N), 0.02, 1.0))
        self.stage = self.infant.stage.values.astype(float)
        self.plus = self.infant.plus.values.astype(float)
        self._hazard = 0.0005 + 0.040 * self.u_true ** 2                  # hidden
        self._arrival_p = 0.15 + 0.25 * self.u_true                       # hidden

    def reset(self):
        self._sample_infants()
        self.t = 0
        self.shadow = self.rng.normal(0, SHADOW_SIGMA, self.N)
        self.battery = np.full(self.N, self.B)
        self.queue_bits = np.zeros(self.N)
        self.queue_tasks = [[] for _ in range(self.N)]
        self.pending_cross = [[] for _ in range(self.N)]
        self.completed_lat, self.cross_lat, self.completion_log = [], [], []
        self.mis_sched = self.cross_served = 0
        self.energy_total = self.sent_frames = self.lost_frames = 0.0
        self.deadline_met = self.deadline_total = 0
        self.alive = np.ones(self.N, bool)
        self.first_death = None
        self.last_arrivals = np.zeros(self.N)
        self.last_cross_detected = np.zeros(self.N)
        self.per = self._channel()
        self.node_reward = np.zeros(self.N)
        return self.obs()

    def _channel(self):
        self.shadow = 0.85 * self.shadow + np.sqrt(1 - 0.85 ** 2) * self.rng.normal(0, SHADOW_SIGMA, self.N)
        pl = PL0_DB + 10 * PL_EXP * np.log10(self.dist) + self.shadow
        interf = (self.rng.uniform(size=self.K) < self.p_int) * 6.0
        snr = TX_POWER_DBM - pl - (NOISE_DBM + self.noise_offset + interf[self.gw])
        return np.clip(per_from_snr_db(snr), 0.005, 0.95)

    def obs(self):
        gl = np.array([self.queue_bits[self.gw == k].sum() for k in range(self.K)]) / 4e6
        wait = np.array([(self.t - q[0]["t0"]) if q else 0.0 for q in self.queue_tasks]) / 10.0
        return np.column_stack([self.queue_bits / 4e6, self.per, self.battery / self.B, wait, gl[self.gw], self.stage / 3.0, self.plus])

    def step(self, bw_share, offload, capture=None):
        bw_share = np.asarray(bw_share, float).copy()
        offload = np.asarray(offload, int)
        capture = np.ones(self.N, int) if capture is None else np.asarray(capture, int)
        self.t += 1
        self.per = self._channel()
        energy = np.zeros(self.N)
        self.last_arrivals = np.zeros(self.N)
        self.last_cross_detected = np.zeros(self.N)
        for i in range(self.N):
            if not self.alive[i]:
                continue
            if self.rng.uniform() < self._hazard[i]:
                self.pending_cross[i].append(self.t)
            if self.rng.uniform() < min(self._arrival_p[i] * CAPTURE_SCALE[capture[i]], 0.95):
                energy[i] += E_CAPTURE_J
                bits = self.rng.uniform(*IMG_BITS_RANGE)
                cross = self.pending_cross[i].pop(0) if self.pending_cross[i] else None
                self.queue_tasks[i].append({"bits": bits, "rem": bits, "t0": self.t, "cross": cross, "u": self.u_true[i]})
                self.queue_bits[i] += bits
                self.last_arrivals[i] = 1.0
        for k in range(self.K):
            m = (self.gw == k) & self.alive
            s = bw_share[m].sum()
            if s > 1.0:
                bw_share[m] = bw_share[m] / s
        f_edge_i = F_EDGE / max(np.sum((offload == 2) & self.alive & (self.queue_bits > 0)), 1)
        served_now = []
        for i in range(self.N):
            if not self.alive[i]:
                continue
            frames = TELEMETRY_BITS / FRAME_BITS
            retx = 1.0 / (1.0 - self.per[i])
            energy[i] += TELEMETRY_BITS * retx * E_TX_J_PER_BIT + 0.1 * TELEMETRY_BITS * E_RX_J_PER_BIT + E_IDLE_ROUND
            self.sent_frames += frames * retx
            self.lost_frames += frames * (retx - 1.0)
            if not self.queue_tasks[i]:
                continue
            o = offload[i]
            if o == 0:
                task = self.queue_tasks[i].pop(0)
                energy[i] += INFER_CYCLES * E_LOCAL_J_PER_CYCLE + RESULT_BITS * retx * E_TX_J_PER_BIT
                self._complete(i, task, (self.t - task["t0"]) * ROUND_SEC + INFER_CYCLES / F_LOCAL, served_now)
                self.queue_bits[i] -= task["bits"]
            else:
                rate = bw_share[i] * RATE_BPS * CSMA_EFF * (1.0 - self.per[i])
                cap, used_s = rate * ROUND_SEC, 0.0
                while self.queue_tasks[i] and cap > 0:
                    task = self.queue_tasks[i][0]
                    send = min(task["rem"], cap)
                    task["rem"] -= send
                    cap -= send
                    used_s += send / max(rate, 1e-9)
                    energy[i] += send * retx * E_TX_J_PER_BIT
                    self.sent_frames += (send / FRAME_BITS) * retx
                    self.lost_frames += (send / FRAME_BITS) * (retx - 1.0)
                    if task["rem"] <= 1e-6:
                        comp = INFER_CYCLES / (F_GW if o == 1 else f_edge_i)
                        self._complete(i, task, (self.t - task["t0"]) * ROUND_SEC + used_s + comp + (0.05 if o == 2 else 0.0), served_now)
                        self.queue_tasks[i].pop(0)
                        self.queue_bits[i] -= task["bits"]
            self.queue_bits[i] = max(self.queue_bits[i], 0.0)
        self.battery -= energy
        self.energy_total += energy.sum()
        newly_dead = self.alive & (self.battery <= 0)
        if newly_dead.any() and self.first_death is None:
            self.first_death = self.t
        self.alive &= ~newly_dead
        if served_now:
            self._mis_scheduling(served_now)
        done = self.t >= self.max_rounds or (not self.alive.any())
        return self.obs(), self._reward(energy, served_now), done

    def _complete(self, i, task, lat, served_now):
        self.completed_lat.append(lat)
        self.deadline_total += 1
        self.deadline_met += int(lat <= DEADLINE_SEC)
        if task["cross"] is not None:
            self.cross_lat.append((self.t - task["cross"]) * ROUND_SEC + (lat - (self.t - task["t0"]) * ROUND_SEC))
            self.cross_served += 1
            self.last_cross_detected[i] += 1
        served_now.append((i, task, lat))

    def _mis_scheduling(self, served_now):
        for i, task, lat in served_now:
            if task["cross"] is not None and any((tc >= task["t0"]) and (u < task["u"]) and (t0 > task["t0"]) for tc, u, t0 in self.completion_log):
                self.mis_sched += 1
        self.completion_log += [(self.t, task["u"], task["t0"]) for _, task, _ in served_now]
        self.completion_log = [c for c in self.completion_log if c[0] > self.t - 200]

    def _reward(self, energy, served_now):
        rn = np.zeros(self.N)
        for i, task, lat in served_now:
            rn[i] += -0.15 * (0.3 + task["u"]) * (lat / DEADLINE_SEC) + (0.5 if lat <= DEADLINE_SEC else -0.5)
            if task["cross"] is not None:
                rn[i] += 4.0 - min((self.t - task["cross"]) / 4.0, 5.0)
        for i in range(self.N):
            if self.queue_tasks[i]:
                rn[i] -= 0.02 * (0.3 + self.u_true[i]) * len(self.queue_tasks[i])
            if self.pending_cross[i]:
                rn[i] -= 1.5 * (0.5 + self.u_true[i]) * len(self.pending_cross[i])
        rn -= 0.25 * energy
        self.node_reward = rn
        return float(rn.sum())

    def metrics(self):
        pdr = 1.0 - self.lost_frames / max(self.sent_frames, 1.0)
        deadline = self.deadline_met / max(self.deadline_total, 1)
        e_node = self.energy_total / max(self.t, 1) / self.N
        eff = np.clip(1.0 - e_node / 0.6, 0, 1)
        fair = 1.0
        if self.completed_lat:
            x = np.array(self.completed_lat)
            fair = (x.sum() ** 2) / (len(x) * np.sum(x ** 2) + 1e-9)
        return {"mean_latency_s": float(np.mean(self.completed_lat)) if self.completed_lat else float("nan"),
                "cross_latency_s": float(np.mean(self.cross_lat)) if self.cross_lat else float("nan"),
                "deadline_rate": float(deadline), "pdr": float(pdr), "energy_per_node_round_mJ": float(1000 * e_node),
                "qos_index": float(100.0 * (0.4 * deadline + 0.3 * pdr + 0.2 * eff + 0.1 * fair)), "fairness": float(fair),
                "mis_sched_rate": float(self.mis_sched / max(self.cross_served, 1)), "n_cross": int(len(self.cross_lat)),
                "first_death_round": self.first_death, "rounds": int(self.t)}

## 7. Baselines (re-implemented under one action space)
The five baselines are re-implementations of the cited 2024 to 2026 methods' core learning mechanisms (heterogeneous graph attention PPO, PPO resource allocation, prioritized noisy dueling style DQN, digital twin assisted QMIX, federated DQN). They are not the authors' original code; state this in the paper.

In [ ]:
N_ACT, OBS_DIM = 27, 8


def decode(a):
    a = np.asarray(a, int)
    return a // 9, (a % 9) // 3, a % 3          # capture level, offload target (0 local, 1 gateway, 2 edge), priority


def pri_to_bw(pri, gw, K, alive=None):
    w = np.array([1.0, 2.0, 4.0])[pri]
    if alive is not None:
        w = w * alive
    return normalize_bw(w, gw, K)


def normalize_bw(w, gw, K):
    bw = np.zeros_like(w, float)
    for k in range(K):
        m = gw == k
        s = w[m].sum()
        bw[m] = w[m] / s if s > 0 else 0.0
    return bw


def node_obs(env, u):
    return np.column_stack([env.obs(), u])


def mlp(i, h, o):
    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, o))


class Agent:
    name, uses = "base", "u_causal"

    def act(self, env, obs, explore=True):
        raise NotImplementedError

    def observe(self, obs, r_node, r_glob, obs2, done, env):
        pass

    def track(self, env):
        pass

    def end_episode(self):
        pass


class PPOBase(Agent):
    def __init__(self, seed, graph=False, lr=3e-4, gamma=0.97, lam=0.95, clip=0.2, epochs=4, h=64):
        torch.manual_seed(seed)
        self.graph, self.h = graph, h
        self.enc = mlp(OBS_DIM, h, h)
        mods = [self.enc]
        if graph:
            self.att_q, self.att_k, self.att_v = (nn.Linear(h, h, bias=False) for _ in range(3))
            self.gw_enc = nn.Linear(3, h)
            self.pi, self.v = mlp(3 * h, h, N_ACT), mlp(3 * h, h, 1)
            mods += [self.att_q, self.att_k, self.att_v, self.gw_enc]
        else:
            self.pi, self.v = mlp(h, h, N_ACT), mlp(h, h, 1)
        mods += [self.pi, self.v]
        self.opt = torch.optim.Adam([p for m in mods for p in m.parameters()], lr=lr)
        self.gamma, self.lam, self.clip, self.epochs = gamma, lam, clip, epochs
        self.ep = []

    def _embed(self, x, gw, gfeat):
        h = F.relu(self.enc(x))
        if not self.graph:
            return h
        q, k, v = self.att_q(h), self.att_k(h), self.att_v(h)
        scores = ((q @ k.T) / math.sqrt(self.h)).masked_fill((gw[:, None] == gw[None, :]) == 0, -1e9)
        return torch.cat([h, torch.softmax(scores, 1) @ v, F.relu(self.gw_enc(gfeat))[gw]], 1)

    def _gfeat(self, env):
        return torch.tensor([[env.queue_bits[env.gw == k].sum() / 8e6, env.per[env.gw == k].mean(), (env.gw == k).sum() / 10.0]
                             for k in range(env.K)], dtype=torch.float32)

    def act(self, env, obs, explore=True):
        x, gw, gf = torch.tensor(obs, dtype=torch.float32), torch.tensor(env.gw), self._gfeat(env)
        with torch.no_grad():
            z = self._embed(x, gw, gf)
            dist = torch.distributions.Categorical(logits=self.pi(z))
            a = dist.sample() if explore else dist.probs.argmax(1)
            self._last = (x, gw, gf, a, dist.log_prob(a), self.v(z).squeeze(-1))
        cap, off, pri = decode(a.numpy())
        return pri_to_bw(pri, env.gw, env.K, env.alive), off, cap

    def observe(self, obs, r_node, r_glob, obs2, done, env):
        x, gw, gf, a, lp, val = self._last
        self.ep.append((x, gw, gf, a, lp, val, torch.tensor(r_node + 0.2 * r_glob / env.N, dtype=torch.float32)))

    def end_episode(self):
        if not self.ep:
            return
        rews, vals = torch.stack([e[6] for e in self.ep]), torch.stack([e[5] for e in self.ep])
        T = len(self.ep)
        adv, last = torch.zeros_like(rews), torch.zeros(rews.shape[1])
        for t in reversed(range(T)):
            nv = vals[t + 1] if t + 1 < T else torch.zeros_like(last)
            last = rews[t] + self.gamma * nv - vals[t] + self.gamma * self.lam * last
            adv[t] = last
        ret = adv + vals
        adv = (adv - adv.mean()) / (adv.std() + 1e-6)
        for _ in range(self.epochs):
            idx = torch.randperm(T)
            for b in range(0, T, 16):
                loss = 0.0
                for t in idx[b: b + 16].tolist():
                    x, gw, gf, a, lp_old = self.ep[t][:5]
                    z = self._embed(x, gw, gf)
                    dist = torch.distributions.Categorical(logits=self.pi(z))
                    ratio = torch.exp(dist.log_prob(a) - lp_old)
                    loss = loss - torch.min(ratio * adv[t], torch.clamp(ratio, 1 - self.clip, 1 + self.clip) * adv[t]).mean() \
                        + 0.5 * F.mse_loss(self.v(z).squeeze(-1), ret[t]) - 0.01 * dist.entropy().mean()
                self.opt.zero_grad()
                (loss / 16).backward()
                self.opt.step()
        self.ep = []


class HGAT_PPO(PPOBase):
    name = "HGAT-PPO"

    def __init__(self, seed):
        super().__init__(seed, graph=True)


class PPO_RA(PPOBase):
    name = "PPO-RA"

    def __init__(self, seed):
        super().__init__(seed, graph=False)


class NoisyLinear(nn.Module):
    def __init__(self, i, o, sigma0=0.5):
        super().__init__()
        self.i, self.o = i, o
        self.w_mu, self.w_sig = nn.Parameter(torch.empty(o, i)), nn.Parameter(torch.full((o, i), sigma0 / math.sqrt(i)))
        self.b_mu, self.b_sig = nn.Parameter(torch.empty(o)), nn.Parameter(torch.full((o,), sigma0 / math.sqrt(i)))
        bound = 1 / math.sqrt(i)
        self.w_mu.data.uniform_(-bound, bound)
        self.b_mu.data.uniform_(-bound, bound)
        self.reset_noise()

    def reset_noise(self):
        f = lambda x: x.sign() * x.abs().sqrt()
        self.eps_i, self.eps_o = f(torch.randn(self.i)), f(torch.randn(self.o))

    def forward(self, x):
        if self.training:
            return F.linear(x, self.w_mu + self.w_sig * torch.outer(self.eps_o, self.eps_i), self.b_mu + self.b_sig * self.eps_o)
        return F.linear(x, self.w_mu, self.b_mu)


class Replay:
    def __init__(self, cap=60000, prioritized=False, alpha=0.6):
        self.cap, self.prio, self.alpha = cap, prioritized, alpha
        self.data, self.p, self.ptr, self.n = [], np.zeros(cap), 0, 0

    def add(self, item, prio=1.0):
        if self.n < self.cap:
            self.data.append(item)
        else:
            self.data[self.ptr] = item
        self.p[self.ptr] = prio ** self.alpha if self.prio else 1.0
        self.ptr, self.n = (self.ptr + 1) % self.cap, min(self.n + 1, self.cap)

    def sample(self, bs, rng):
        probs = self.p[: self.n] / self.p[: self.n].sum()
        idx = rng.choice(self.n, bs, replace=self.n < bs, p=probs)
        w = (self.n * probs[idx]) ** (-0.4)
        return idx, [self.data[i] for i in idx], w / w.max()

    def update(self, idx, prios):
        self.p[idx] = np.asarray(prios) ** self.alpha


class DQNBase(Agent):
    def __init__(self, seed, noisy=False, prioritized=False, lr=5e-4, gamma=0.97, h=64, bs=128, target_every=200):
        torch.manual_seed(seed)
        self.rng, self.noisy = np.random.default_rng(seed), noisy
        L = NoisyLinear if noisy else nn.Linear
        self.q = nn.Sequential(nn.Linear(OBS_DIM, h), nn.ReLU(), L(h, h), nn.ReLU(), L(h, N_ACT))
        self.qt = copy.deepcopy(self.q)
        self.opt = torch.optim.Adam(self.q.parameters(), lr=lr)
        self.buf = Replay(prioritized=prioritized)
        self.gamma, self.bs, self.target_every, self.steps, self.eps = gamma, bs, target_every, 0, 1.0

    def _noise(self, net):
        for m in net.modules():
            if isinstance(m, NoisyLinear):
                m.reset_noise()

    def act(self, env, obs, explore=True):
        self.q.train(explore and self.noisy)
        with torch.no_grad():
            if self.noisy and explore:
                self._noise(self.q)
            a = self.q(torch.tensor(obs, dtype=torch.float32)).argmax(1).numpy()
        if explore and not self.noisy:
            m = self.rng.uniform(size=len(a)) < self.eps
            a[m] = self.rng.integers(0, N_ACT, m.sum())
        self._last_a = a
        cap, off, pri = decode(a)
        return pri_to_bw(pri, env.gw, env.K, env.alive), off, cap

    def _prio(self, obs, r):
        return np.ones(len(obs))

    def observe(self, obs, r_node, r_glob, obs2, done, env):
        pr = self._prio(obs, r_node)
        for i in range(env.N):
            self.buf.add((obs[i], self._last_a[i], r_node[i], obs2[i], float(done)), pr[i])
        self.steps += 1
        self.eps = max(0.05, 1.0 - self.steps / 6000.0)
        if self.buf.n >= 1000:
            self._learn()
        if self.steps % self.target_every == 0:
            self.qt.load_state_dict(self.q.state_dict())

    def _learn(self):
        idx, batch, w = self.buf.sample(self.bs, self.rng)
        o = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32)
        a = torch.tensor([b[1] for b in batch])
        r = torch.tensor([b[2] for b in batch], dtype=torch.float32)
        o2 = torch.tensor(np.array([b[3] for b in batch]), dtype=torch.float32)
        d = torch.tensor([b[4] for b in batch], dtype=torch.float32)
        self.q.train(True)
        if self.noisy:
            self._noise(self.q)
            self._noise(self.qt)
        qsa = self.q(o).gather(1, a[:, None]).squeeze(1)
        with torch.no_grad():
            tgt = r + self.gamma * (1 - d) * self.qt(o2).gather(1, self.q(o2).argmax(1)[:, None]).squeeze(1)
        td = tgt - qsa
        loss = (torch.tensor(w, dtype=torch.float32) * td.pow(2)).mean()
        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 5.0)
        self.opt.step()
        if self.buf.prio:
            self.buf.update(idx, np.abs(td.detach().numpy()) + 1e-3)


class PD_NoisyDQN(DQNBase):
    name = "PD-NoisyDQN"

    def __init__(self, seed):
        super().__init__(seed, noisy=True, prioritized=True)

    def _prio(self, obs, r):
        return (1.0 + 2.0 * obs[:, -1]) * (np.abs(r) + 0.1)


class JOFO_FedDQN(Agent):
    name = "JOFO-FedDQN"

    def __init__(self, seed, K=3, fed_every=20):
        self.locals = [DQNBase(seed + 11 * k) for k in range(K)]
        self.K, self.fed_every, self.steps, self.counts = K, fed_every, 0, np.zeros(K)

    def act(self, env, obs, explore=True):
        a = np.zeros(env.N, int)
        for k in range(self.K):
            m = env.gw == k
            if m.any():
                ag = self.locals[k]
                with torch.no_grad():
                    ak = ag.q(torch.tensor(obs[m], dtype=torch.float32)).argmax(1).numpy()
                if explore:
                    mm = ag.rng.uniform(size=len(ak)) < ag.eps
                    ak[mm] = ag.rng.integers(0, N_ACT, mm.sum())
                a[m] = ak
        self._last_a = a
        cap, off, pri = decode(a)
        return pri_to_bw(pri, env.gw, env.K, env.alive), off, cap

    def observe(self, obs, r_node, r_glob, obs2, done, env):
        for k in range(self.K):
            ag = self.locals[k]
            for i in np.where(env.gw == k)[0]:
                ag.buf.add((obs[i], self._last_a[i], r_node[i], obs2[i], float(done)))
            self.counts[k] += (env.gw == k).sum()
            ag.steps += 1
            ag.eps = max(0.05, 1.0 - ag.steps / 6000.0)
            if ag.buf.n >= 1000:
                ag._learn()
            if ag.steps % ag.target_every == 0:
                ag.qt.load_state_dict(ag.q.state_dict())
        self.steps += 1
        if self.steps % self.fed_every == 0:
            w = self.counts / max(self.counts.sum(), 1)
            sd = [ag.q.state_dict() for ag in self.locals]
            avg = {k: sum(w[j] * sd[j][k] for j in range(self.K)) for k in sd[0]}
            for ag in self.locals:
                ag.q.load_state_dict(avg)
            self.counts[:] = 0


class Mixer(nn.Module):
    def __init__(self, state_dim=4, h=32):
        super().__init__()
        self.hw1, self.hb1, self.hw2 = nn.Linear(state_dim, h), nn.Linear(state_dim, h), nn.Linear(state_dim, h)
        self.hb2 = nn.Sequential(nn.Linear(state_dim, h), nn.ReLU(), nn.Linear(h, 1))
        self.att = nn.Linear(OBS_DIM, 1)
        self.h = h

    def forward(self, qs, obs, state):
        q = (qs * torch.softmax(self.att(obs).squeeze(-1), -1) * qs.shape[-1]).unsqueeze(1)
        w1 = torch.abs(self.hw1(state)).view(-1, 1, self.h).expand(q.shape[0], qs.shape[-1], self.h)
        hidden = F.elu(torch.bmm(q, w1) + self.hb1(state).view(-1, 1, self.h))
        return (torch.bmm(hidden, torch.abs(self.hw2(state)).view(-1, self.h, 1)) + self.hb2(state).view(-1, 1, 1)).view(-1)


class DT_QMIX(Agent):
    name = "DT-QMIX"

    def __init__(self, seed, lr=5e-4, gamma=0.97, h=64, bs=16):
        torch.manual_seed(seed)
        self.rng = np.random.default_rng(seed)
        self.q, self.mix = mlp(OBS_DIM, h, N_ACT), Mixer()
        self.qt, self.mixt = copy.deepcopy(self.q), copy.deepcopy(self.mix)
        self.twin = mlp(OBS_DIM + N_ACT, h, OBS_DIM + 1)
        self.opt = torch.optim.Adam(list(self.q.parameters()) + list(self.mix.parameters()), lr=lr)
        self.topt = torch.optim.Adam(self.twin.parameters(), lr=1e-3)
        self.buf, self.gamma, self.bs, self.steps, self.eps = [], gamma, bs, 0, 1.0

    def _state(self, env):
        return np.array([env.queue_bits.sum() / 8e7, env.per.mean(), env.battery.mean() / env.B, env.alive.mean()], dtype=np.float32)

    def act(self, env, obs, explore=True):
        with torch.no_grad():
            a = self.q(torch.tensor(obs, dtype=torch.float32)).argmax(1).numpy()
        if explore:
            m = self.rng.uniform(size=len(a)) < self.eps
            a[m] = self.rng.integers(0, N_ACT, m.sum())
        self._last_a, self._last_s = a, self._state(env)
        cap, off, pri = decode(a)
        return pri_to_bw(pri, env.gw, env.K, env.alive), off, cap

    def observe(self, obs, r_node, r_glob, obs2, done, env):
        self.buf.append((obs.astype(np.float32), self._last_a.copy(), float(r_glob), obs2.astype(np.float32), float(done),
                         self._last_s, self._state(env), r_node.astype(np.float32)))
        if len(self.buf) > 20000:
            self.buf.pop(0)
        self.steps += 1
        self.eps = max(0.05, 1.0 - self.steps / 6000.0)
        if len(self.buf) >= 200 and self.steps % 2 == 0:
            self._learn_twin()
            self._learn()
            if self.steps % 6 == 0:
                self._learn(twin=True)
        if self.steps % 200 == 0:
            self.qt.load_state_dict(self.q.state_dict())
            self.mixt.load_state_dict(self.mix.state_dict())

    def _learn_twin(self):
        xs, ys = [], []
        for i in self.rng.integers(0, len(self.buf), 8):
            o, a, r, o2, d, s, s2, rn = self.buf[i]
            xs.append(np.concatenate([o, np.eye(N_ACT)[a]], 1))
            ys.append(np.concatenate([o2, rn[:, None]], 1))
        loss = F.mse_loss(self.twin(torch.tensor(np.concatenate(xs), dtype=torch.float32)), torch.tensor(np.concatenate(ys), dtype=torch.float32))
        self.topt.zero_grad()
        loss.backward()
        self.topt.step()

    def _learn(self, twin=False):
        loss = 0.0
        for i in self.rng.integers(0, len(self.buf), self.bs):
            o, a, r, o2, d, s, s2, rn = self.buf[i]
            o_t, a_t = torch.tensor(o), torch.tensor(a)
            if twin:
                a_alt = self.rng.integers(0, N_ACT, len(a))
                with torch.no_grad():
                    pred = self.twin(torch.cat([o_t, torch.eye(N_ACT)[a_alt]], 1))
                o2_t, r, a_t = pred[:, :OBS_DIM], float(pred[:, OBS_DIM].sum()), torch.tensor(a_alt)
            else:
                o2_t = torch.tensor(o2)
            qtot = self.mix(self.q(o_t).gather(1, a_t[:, None]).squeeze(1)[None], o_t[None], torch.tensor(s)[None])
            with torch.no_grad():
                q2 = self.qt(o2_t).gather(1, self.q(o2_t).argmax(1)[:, None]).squeeze(1)
                tgt = r + self.gamma * (1 - d) * self.mixt(q2[None], o2_t[None], torch.tensor(s2)[None])
            loss = loss + F.mse_loss(qtot, tgt)
        self.opt.zero_grad()
        (loss / self.bs).backward()
        nn.utils.clip_grad_norm_(list(self.q.parameters()) + list(self.mix.parameters()), 5.0)
        self.opt.step()

## 8. InterventionROP-Edge controller
1. Counterfactual threshold urgency from the stratified causal estimator (last observation column).
2. Candidate actions: a structured grid (bandwidth weighting, offloading, capture rate) plus one candidate from gradient ascent on a differentiable QoS surrogate learned from observed per node rewards.
3. Digital twin: predicts the operator's QoS objective and per node latency under do(action), using only observable quantities (queues, link PER estimate, online arrival and crossing rate estimates). The hidden arrival and hazard parameters of the simulator are never read.
4. Do-calculus constraint: (a) bandwidth is reassigned so that it is monotone in the interventional urgency within each gateway; (b) high risk infants keep at least normal imaging; (c) candidates whose predicted latency under do(action) exceeds the deadline for threshold risk infants are rejected (minimum violation fallback).

In [ ]:
class InterventionROP(Agent):
    name, uses = "InterventionROP-Edge", "u_causal"

    def __init__(self, seed, use_constraint=True, use_twin=True, use_surrogate=True, lr=1e-3, h=96, lam_e=0.25,
                 u_crit=0.15, opt_steps=25):     # u_crit 0.15 = top ~10% of the cohort (threshold risk infants)
        torch.manual_seed(seed)
        self.rng = np.random.default_rng(seed)
        self.use_constraint, self.use_twin, self.use_surrogate = use_constraint, use_twin, use_surrogate
        self.sur = mlp(OBS_DIM + 1 + 3 + 3, h, 1)
        self.opt = torch.optim.Adam(self.sur.parameters(), lr=lr)
        self.buf_x, self.buf_y = [], []
        self.lam_e, self.u_crit, self.opt_steps, self.t = lam_e, u_crit, opt_steps, 0
        self.cross_seen, self.node_rounds = 0.0, 0.0
        self._env, self.prev = None, None

    # online estimators (observable information only)
    def _new_env(self, env):
        self._env = env
        self.per_est = env.per.copy()
        self.arr_hat = np.full(env.N, 0.25)
        self.cap_used = np.ones(env.N, int)
        self.prev = None

    def track(self, env):
        self.per_est = 0.8 * self.per_est + 0.2 * env.per
        base = env.last_arrivals / CAPTURE_SCALE[self.cap_used]
        self.arr_hat = np.where(env.alive, 0.95 * self.arr_hat + 0.05 * base, self.arr_hat)
        self.cross_seen += env.last_cross_detected.sum()
        self.node_rounds += env.alive.sum()

    def _hazard(self, u):
        lam = (self.cross_seen + 1.0) / (self.node_rounds + 200.0)
        return np.clip(lam * u / max(u.mean(), 1e-3), 0.0, 0.5)

    def _twin(self, env, u, bw, off, cap):
        per, q = self.per_est, env.queue_bits
        mean_bits = np.mean(IMG_BITS_RANGE)
        exp_arr = np.minimum(self.arr_hat * CAPTURE_SCALE[cap], 0.95) * env.alive
        load = q + exp_arr * mean_bits
        thr = bw * RATE_BPS * CSMA_EFF * (1.0 - per) * ROUND_SEC
        n_edge = max(1, int(((off == 2) & env.alive).sum()))
        f_off = np.where(off == 1, F_GW, F_EDGE / n_edge)
        lat = np.where(off == 0, (q / mean_bits + exp_arr) * ROUND_SEC + INFER_CYCLES / F_LOCAL,
                       load / np.maximum(thr, 1e3) * ROUND_SEC + INFER_CYCLES / f_off)
        e = exp_arr * E_CAPTURE_J + np.where(off == 0, exp_arr * INFER_CYCLES * E_LOCAL_J_PER_CYCLE,
                                              np.minimum(load, thr) * E_TX_J_PER_BIT / (1.0 - per))
        hz = self._hazard(u)
        wait = 1.0 / np.maximum(exp_arr, 1e-3) + lat / ROUND_SEC
        util = exp_arr * (-0.15 * (0.3 + u) * lat / DEADLINE_SEC + np.where(lat <= DEADLINE_SEC, 0.5, -0.5))
        util = util - 1.5 * (0.5 + u) * hz * wait + hz * (4.0 - np.minimum(wait / 4.0, 5.0))
        util = util - 0.02 * (0.3 + u) * (q > 0) - self.lam_e * e
        return float((util * env.alive).sum()), lat

    def _project(self, env, u, bw, off, cap):
        if not self.use_constraint:
            return bw, off, cap
        bw, off, cap = bw.copy(), off.copy(), cap.copy()
        hi = u >= self.u_crit
        cap[hi] = np.maximum(cap[hi], 1)
        for k in range(env.K):
            m = np.where((env.gw == k) & env.alive)[0]
            if len(m):
                bw[m[np.argsort(-u[m])]] = np.sort(bw[m])[::-1]
        return bw, off, cap

    def _grid(self, env, u):
        N, out = env.N, []
        for p in (0.0, 1.0, 2.0):
            bw = normalize_bw((0.25 + u ** p if p > 0 else np.ones(N)) * env.alive, env.gw, env.K)
            for off in (np.full(N, 2), np.where(u > 0.3, 2, 1), np.full(N, 1)):
                for cap in (np.where(u >= 0.3, 1, 0), np.ones(N, int), np.where(u >= self.u_crit, 2, 1)):
                    out.append((bw, off.astype(int), cap.astype(int)))
        return out

    def _surrogate_candidate(self, env, obs, u):
        N = env.N
        x = torch.tensor(obs, dtype=torch.float32)
        logit_bw = torch.tensor(np.log(0.25 + u ** 2), dtype=torch.float32, requires_grad=True)
        logit_off = torch.zeros(N, 3)
        logit_off[:, 2] += 0.5
        logit_cap = torch.zeros(N, 3)
        logit_cap[:, 1] += 0.5
        logit_off.requires_grad_(True)
        logit_cap.requires_grad_(True)
        opt = torch.optim.Adam([logit_bw, logit_off, logit_cap], lr=0.15)
        gw, u_t = torch.tensor(env.gw), torch.tensor(u, dtype=torch.float32)
        for _ in range(self.opt_steps):
            bw = sum(torch.where(gw == k, torch.softmax(logit_bw.masked_fill(gw != k, -1e9), 0), torch.zeros(N)) for k in range(env.K))
            po, pc = torch.softmax(logit_off, 1), torch.softmax(logit_cap, 1)
            util = self.sur(torch.cat([x, bw[:, None], po, pc], 1)).squeeze(-1)
            loss = -((0.5 + u_t) * util).sum() + 0.01 * (po * torch.log(po + 1e-8)).sum() + 0.01 * (pc * torch.log(pc + 1e-8)).sum()
            opt.zero_grad()
            loss.backward()
            opt.step()
        with torch.no_grad():
            return bw.numpy().copy() * env.alive, logit_off.argmax(1).numpy(), logit_cap.argmax(1).numpy()

    def act(self, env, obs, explore=True):
        if self._env is not env:
            self._new_env(env)
        u = obs[:, -1]
        cand = self._grid(env, u)
        sur_ready = self.use_surrogate and len(self.buf_x) > 1500
        if sur_ready:
            cand.append(self._surrogate_candidate(env, obs, u))
        if self.prev is not None:
            cand.append(self.prev)
        cand = [self._project(env, u, *c) for c in cand]
        if self.use_twin:
            ev = [self._twin(env, u, *c) for c in cand]
            util = np.array([e[0] for e in ev])
            if self.use_constraint:
                crit = (u >= self.u_crit) & env.alive & (env.queue_bits > 0)
                viol = np.array([np.maximum(e[1][crit] - DEADLINE_SEC, 0.0).sum() for e in ev])
                ok = viol <= 1e-9
                best = int(np.argmax(np.where(ok, util, -np.inf))) if ok.any() else int(np.lexsort((-util, viol))[0])
            else:
                best = int(np.argmax(util))
        else:
            best = len(self._grid(env, u)) if sur_ready else 13      # surrogate candidate, else grid default (u-weighted, split, normal)
        bw, off, cap = [np.array(v) for v in cand[best]]
        if explore and self.rng.uniform() < max(0.05, 0.6 - self.t / 4000.0):
            m = self.rng.uniform(size=env.N) < 0.3
            off[m] = self.rng.integers(0, 3, m.sum())
            cap[m] = self.rng.integers(0, 3, m.sum())
            bw = normalize_bw(bw * self.rng.uniform(0.5, 1.5, env.N) * env.alive, env.gw, env.K)
        self.prev = (bw.copy(), off.copy(), cap.copy())
        self.cap_used = cap.copy()
        self._last = (obs.copy(), bw.copy(), off.copy(), cap.copy())
        return bw, off, cap

    def observe(self, obs, r_node, r_glob, obs2, done, env):
        o, bw, off, cap = self._last
        self.buf_x.extend(list(np.concatenate([o, bw[:, None], np.eye(3)[off], np.eye(3)[cap]], 1).astype(np.float32)))
        self.buf_y.extend(list(r_node.astype(np.float32)))
        if len(self.buf_x) > 60000:
            self.buf_x, self.buf_y = self.buf_x[-60000:], self.buf_y[-60000:]
        self.t += 1
        if len(self.buf_x) >= 500 and self.t % 2 == 0:
            idx = self.rng.integers(0, len(self.buf_x), 256)
            loss = F.smooth_l1_loss(self.sur(torch.tensor(np.array([self.buf_x[i] for i in idx]))).squeeze(-1),
                                    torch.tensor(np.array([self.buf_y[i] for i in idx])))
            self.opt.zero_grad()
            loss.backward()
            self.opt.step()


def make_agent(name, seed):
    table = {"InterventionROP-Edge": lambda: InterventionROP(seed), "HGAT-PPO": lambda: HGAT_PPO(seed),
             "PPO-RA": lambda: PPO_RA(seed), "PD-NoisyDQN": lambda: PD_NoisyDQN(seed), "DT-QMIX": lambda: DT_QMIX(seed),
             "JOFO-FedDQN": lambda: JOFO_FedDQN(seed),
             "Abl-NoDoConstraint": lambda: InterventionROP(seed, use_constraint=False),
             "Abl-NoTwin": lambda: InterventionROP(seed, use_twin=False),
             "Abl-NoSurrogate": lambda: InterventionROP(seed, use_surrogate=False),
             "Abl-CorrUrgency": lambda: InterventionROP(seed), "Abl-Unstratified": lambda: InterventionROP(seed)}
    ag = table[name]()
    ag.name = name
    ag.uses = {"Abl-CorrUrgency": "u_corr", "Abl-Unstratified": "u_unstrat"}.get(name, "u_causal")   # same urgency for every method
    return ag

## 9. Training and evaluation

In [ ]:
def run_episode(agent, env, explore, collect=False):
    env.reset()
    u = getattr(env, agent.uses)
    obs, done, R = node_obs(env, u), False, 0.0
    while not done:
        bw, off, cap = agent.act(env, obs, explore)
        _, r, done = env.step(bw, off, cap)
        agent.track(env)
        obs2 = node_obs(env, u)
        if explore:
            agent.observe(obs, env.node_reward, r, obs2, done, env)
        obs, R = obs2, R + r
    if explore:
        agent.end_episode()
    m = env.metrics()
    m["return"] = R
    if collect:
        m["latencies"] = [float(x) for x in env.completed_lat]
    return m


def train(name, seed=SEED):
    agent, curve = make_agent(name, seed), []
    for ep in range(TRAIN_EPISODES):
        m = run_episode(agent, NICUEnv(20, 3, seed=1000 * seed + ep, cohort=cohort, max_rounds=TRAIN_ROUNDS), explore=True)
        curve.append(m["return"])
        if ep % 20 == 0 or ep == TRAIN_EPISODES - 1:
            log(f"  {name} ep {ep} return={m['return']:.1f} crossLat={m['cross_latency_s']:.0f}s QoS={m['qos_index']:.1f}")
    return agent, curve


def evaluate(agent, n=20, noise=0.0, shift=None, seeds=EVAL_SEEDS, rounds=EVAL_ROUNDS, battery=600.0, collect=False):
    return [run_episode(agent, NICUEnv(n, 3, seed=s, cohort=cohort, max_rounds=rounds, noise_offset_db=noise, shift=shift,
                                       battery_j=battery), explore=False, collect=collect) for s in seeds]


AGG_KEYS = ("cross_latency_s", "mean_latency_s", "energy_per_node_round_mJ", "qos_index", "pdr", "deadline_rate", "mis_sched_rate", "fairness", "return")


def agg(ms):
    return {k: {"mean": float(np.nanmean([m[k] for m in ms])), "std": float(np.nanstd([m[k] for m in ms]))} for k in AGG_KEYS}

In [ ]:
t0 = time.time()
AGENTS, RESULTS["convergence"], RESULTS["default"], RESULTS["per_seed"], RESULTS["latency_samples"] = {}, {}, {}, {}, {}
for name in METHODS + ABLATIONS:
    log("train", name)
    AGENTS[name], RESULTS["convergence"][name] = train(name)
    ms = evaluate(AGENTS[name], collect=True)
    RESULTS["default"][name] = agg(ms)
    RESULTS["per_seed"][name] = [{k: m[k] for k in AGG_KEYS} for m in ms]
    lat = np.concatenate([m["latencies"] for m in ms])
    RESULTS["latency_samples"][name] = [float(x) for x in np.random.default_rng(0).choice(lat, min(len(lat), 3000), replace=False)]
    d = RESULTS["default"][name]
    log(f"  eval {name}: crossLat={d['cross_latency_s']['mean']:.1f}s E={d['energy_per_node_round_mJ']['mean']:.1f}mJ "
        f"QoS={d['qos_index']['mean']:.2f} mis={d['mis_sched_rate']['mean']:.3f}")
json.dump(RESULTS, open(OUT_DIR / "results.json", "w"), indent=1)
log(f"training + default evaluation done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
log("node, channel, GA shift sweeps and lifetime")
RESULTS["node_sweep"] = {"n_nodes": NODE_LIST}
RESULTS["channel_sweep"] = {"noise_offset_db": NOISE_LIST}
RESULTS["ga_shift"], RESULTS["lifetime"] = {}, {}
for name in METHODS:
    ag = AGENTS[name]
    RESULTS["node_sweep"][name] = {str(n): agg(evaluate(ag, n=n, seeds=SWEEP_SEEDS)) for n in NODE_LIST}
    RESULTS["channel_sweep"][name] = {str(z): agg(evaluate(ag, noise=z, seeds=SWEEP_SEEDS)) for z in NOISE_LIST}
    RESULTS["ga_shift"][name] = agg(evaluate(ag, shift="ga_shift"))
    vals = []
    for s in LIFE_SEEDS:
        m = run_episode(ag, NICUEnv(20, 3, seed=s, cohort=cohort, max_rounds=LIFE_MAX_ROUNDS, battery_j=LIFE_BATTERY_J), explore=False)
        vals.append(m["first_death_round"] if m["first_death_round"] is not None else m["rounds"])
    RESULTS["lifetime"][name] = {"mean": float(np.mean(vals)), "std": float(np.std(vals)), "values": [int(v) for v in vals]}
    log(f"  {name}: lifetime {vals}")
json.dump(RESULTS, open(OUT_DIR / "results.json", "w"), indent=1)

## 10. Results tables (report these numbers exactly)

In [ ]:
def table(section, names, keys=("cross_latency_s", "energy_per_node_round_mJ", "qos_index", "pdr", "deadline_rate", "mis_sched_rate")):
    return pd.DataFrame({n: {k: f"{RESULTS[section][n][k]['mean']:.3f} ± {RESULTS[section][n][k]['std']:.3f}" for k in keys} for n in names}).T


print("Default scenario (N = 20)")
print(table("default", METHODS).to_string())
print("\nAblation")
print(table("default", ["InterventionROP-Edge"] + ABLATIONS).to_string())
print("\nGA shifted cohort")
print(table("ga_shift", METHODS).to_string())
print("\nLifetime (rounds to first node depletion, battery %.0f J)" % LIFE_BATTERY_J)
print({n: RESULTS["lifetime"][n] for n in METHODS})

LOWER_BETTER = {"cross_latency_s": True, "energy_per_node_round_mJ": True, "mis_sched_rate": True,
                "qos_index": False, "pdr": False, "deadline_rate": False}
summary = {}
ours = RESULTS["default"]["InterventionROP-Edge"]
for k, low in LOWER_BETTER.items():
    vals = {b: RESULTS["default"][b][k]["mean"] for b in BASELINES}
    best_b = min(vals, key=vals.get) if low else max(vals, key=vals.get)
    o = ours[k]["mean"]
    rel_best = 100.0 * (o - vals[best_b]) / max(abs(vals[best_b]), 1e-9)
    rel_mean = 100.0 * (o - np.mean(list(vals.values()))) / max(abs(np.mean(list(vals.values()))), 1e-9)
    better = (rel_best < 0) if low else (rel_best > 0)
    a, b = [m[k] for m in RESULTS["per_seed"]["InterventionROP-Edge"]], [m[k] for m in RESULTS["per_seed"][best_b]]
    try:
        p = float(wilcoxon(a, b).pvalue)
    except Exception:
        p = float("nan")
    summary[k] = {"ours": o, "best_baseline": best_b, "best_baseline_value": vals[best_b],
                  "change_vs_best_%": rel_best, "change_vs_mean_baselines_%": rel_mean, "ours_better_than_best": bool(better), "wilcoxon_p": p}
lt = {n: RESULTS["lifetime"][n]["mean"] for n in METHODS}
best_lt = max((b for b in BASELINES), key=lambda b: lt[b])
summary["lifetime_rounds"] = {"ours": lt["InterventionROP-Edge"], "best_baseline": best_lt, "best_baseline_value": lt[best_lt],
                              "change_vs_best_%": 100.0 * (lt["InterventionROP-Edge"] / lt[best_lt] - 1),
                              "ours_better_than_best": bool(lt["InterventionROP-Edge"] > lt[best_lt])}
C = RESULTS["causal"]
summary["attribution"] = {k: C[f"attribution_{k}"] for k in list(ESTIMATORS) + ["oracle"]}
RESULTS["summary"] = summary
print("\nProposed vs best baseline (negative % = lower value)")
print(pd.DataFrame({k: v for k, v in summary.items() if k != "attribution"}).T.to_string())
print("\nAttribution:", json.dumps(summary["attribution"], indent=1))
json.dump(RESULTS, open(OUT_DIR / "results.json", "w"), indent=1)


def phrase(k, label):
    s = summary[k]
    low = LOWER_BETTER.get(k, False)
    ch = s["change_vs_best_%"]
    word = ("reduces" if ch < 0 else "increases") if low else ("improves" if ch > 0 else "lowers")
    return f"{word} {label} by {abs(ch):.1f}% relative to the best baseline ({s['best_baseline']})"


print("\nAbstract sentence generated from this run:")
print(f"InterventionROP-Edge {phrase('cross_latency_s', 'threshold-crossing latency')}, "
      f"{phrase('energy_per_node_round_mJ', 'energy consumption')}, and {phrase('qos_index', 'the composite QoS index')}, "
      f"with {100 * C['attribution_causal']['precision']:.1f}% counterfactual attribution precision "
      f"(AUROC {C['attribution_causal']['auroc']:.3f}) and a mean network lifetime of {lt['InterventionROP-Edge']:.0f} monitoring rounds.")

## 11. Results figures (IEEE single column, Times New Roman, 800 dpi)

In [ ]:
avail = {f.name for f in font_manager.fontManager.ttflist}
FONT = next((f for f in ["Times New Roman", "Nimbus Roman", "Liberation Serif", "DejaVu Serif"] if f in avail), "serif")
print("Figure font:", FONT)
plt.rcParams.update({"font.family": "serif", "font.serif": [FONT], "mathtext.fontset": "stix", "font.size": 8,
                     "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
                     "axes.linewidth": 0.6, "lines.linewidth": 1.2, "lines.markersize": 4, "savefig.dpi": 800})
COL = {"InterventionROP-Edge": "#1f4e9c", "HGAT-PPO": "#d55e00", "PPO-RA": "#009e73", "PD-NoisyDQN": "#cc79a7",
       "DT-QMIX": "#8c6d31", "JOFO-FedDQN": "#56b4e9"}
MRK = {"InterventionROP-Edge": "o", "HGAT-PPO": "s", "PPO-RA": "^", "PD-NoisyDQN": "D", "DT-QMIX": "v", "JOFO-FedDQN": "P"}
W, H = 3.5, 2.5


def finish(fig, ax, name, ncol=3, legend=True):
    if legend:
        h, l = ax.get_legend_handles_labels()
        fig.legend(h, l, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=ncol, frameon=False,
                   columnspacing=0.9, handlelength=1.6, borderaxespad=0.0)
    fig.tight_layout(pad=0.3)
    fig.savefig(FIG_DIR / name, dpi=800, bbox_inches="tight", pad_inches=0.03)
    plt.show()
    plt.close(fig)


def bars(ax, labels, means, errs, colors):
    x = np.arange(len(labels))
    ax.bar(x, means, yerr=errs, color=colors, width=0.62, capsize=2.5, error_kw={"elinewidth": 0.7})
    ax.set_xticks(x)
    ax.set_xticklabels([l.replace("-", "-\n", 1) for l in labels])


def sweep_plot(section, xkey, metric, xlabel, ylabel, fname):
    fig, ax = plt.subplots(figsize=(W, H))
    xs = RESULTS[section][xkey]
    for n in METHODS:
        ax.errorbar(xs, [RESULTS[section][n][str(x)][metric]["mean"] for x in xs], yerr=[RESULTS[section][n][str(x)][metric]["std"] for x in xs],
                    color=COL[n], marker=MRK[n], capsize=2, label=n, elinewidth=0.6)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks(xs)
    ax.grid(alpha=0.25, lw=0.4)
    finish(fig, ax, fname)


# Fig. 1: ATE by group
G = list(C["ate_groups"])
fig, ax = plt.subplots(figsize=(W, H + 0.3))
y = np.arange(len(G))
for j, (k, lab, c) in enumerate([("causal", EST_LABEL["causal"], "#1f4e9c"), ("corr", EST_LABEL["corr"], "#d55e00"), ("unstrat", EST_LABEL["unstrat"], "#009e73")]):
    m = np.array([C["ate_groups"][g][k]["mean"] for g in G])
    lo = np.clip(np.nan_to_num(m - np.array([C["ate_groups"][g][k]["lo"] for g in G])), 0, None)
    hi = np.clip(np.nan_to_num(np.array([C["ate_groups"][g][k]["hi"] for g in G]) - m), 0, None)
    ax.errorbar(m, y + (j - 1) * 0.22, xerr=[lo, hi], fmt="o", color=c, capsize=2, label=lab, ms=3.5, elinewidth=0.7)
ax.scatter([C["ate_groups"][g]["true"] for g in G], y, marker="*", s=40, color="black", zorder=5, label="True ATE")
ax.set_yticks(y)
ax.set_yticklabels(G)
ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("Average treatment effect on regression")
ax.grid(axis="x", alpha=0.25, lw=0.4)
finish(fig, ax, "fig_results_01_ate_groups.png", ncol=2)

# Fig. 2: ITE by ICROP stage
fig, ax = plt.subplots(figsize=(W, H))
st = list(C["ite_by_stage"])
pos = np.arange(len(st))
b1 = ax.boxplot([C["ite_by_stage"][s]["est"] for s in st], positions=pos - 0.18, widths=0.3, patch_artist=True, showfliers=False)
b2 = ax.boxplot([C["ite_by_stage"][s]["true"] for s in st], positions=pos + 0.18, widths=0.3, patch_artist=True, showfliers=False)
for b, c in ((b1, "#9db7e0"), (b2, "#d9d9d9")):
    for p in b["boxes"]:
        p.set_facecolor(c)
        p.set_linewidth(0.6)
ax.plot([], [], "s", color="#9db7e0", label="Estimated ITE (proposed)")
ax.plot([], [], "s", color="#d9d9d9", label="True ITE")
ax.set_xticks(pos)
ax.set_xticklabels([f"Stage {s}" for s in st])
ax.set_ylabel("Individual treatment effect")
ax.grid(axis="y", alpha=0.25, lw=0.4)
finish(fig, ax, "fig_results_02_ite_by_stage.png", ncol=2)

# Fig. 3 and 4: confounding sweep (PEHE, urgency rank)
for metric, ylab, fname in (("pehe", "PEHE", "fig_results_03_pehe_vs_confounding.png"),
                            ("urank", "Urgency rank correlation", "fig_results_04_urgency_rank_vs_confounding.png"),
                            ("auroc", "Attribution AUROC", "fig_results_05_attribution_vs_confounding.png")):
    fig, ax = plt.subplots(figsize=(W, H))
    for k, c, mk in (("causal", "#1f4e9c", "o"), ("corr", "#d55e00", "s"), ("unstrat", "#009e73", "^")):
        ax.errorbar(sweep["gamma"], sweep[f"{metric}_{k}"], yerr=sweep[f"{metric}_{k}_sd"], color=c, marker=mk, capsize=2, label=EST_LABEL[k], elinewidth=0.6)
    if metric == "auroc":
        ax.plot(sweep["gamma"], sweep["auroc_oracle"], "k--", lw=0.9, label="Oracle")
    ax.set_xlabel(r"Oxygen confounding strength $\gamma$")
    ax.set_ylabel(ylab)
    ax.grid(alpha=0.25, lw=0.4)
    finish(fig, ax, fname, ncol=2)

# Fig. 6: convergence
fig, ax = plt.subplots(figsize=(W, H))
for n in METHODS:
    c = np.array(RESULTS["convergence"][n])
    k = max(1, len(c) // 10)
    ax.plot(np.arange(len(c)), pd.Series(c).rolling(k, min_periods=1).mean(), color=COL[n], label=n)
ax.set_xlabel("Training episode")
ax.set_ylabel("Episode return")
ax.grid(alpha=0.25, lw=0.4)
finish(fig, ax, "fig_results_06_convergence.png")

sweep_plot("node_sweep", "n_nodes", "cross_latency_s", "Number of NICU sensor nodes", "Threshold-crossing latency (s)", "fig_results_07_cross_latency_vs_nodes.png")
sweep_plot("node_sweep", "n_nodes", "energy_per_node_round_mJ", "Number of NICU sensor nodes", "Energy per node per round (mJ)", "fig_results_08_energy_vs_nodes.png")
sweep_plot("channel_sweep", "noise_offset_db", "qos_index", "Noise floor offset (dB)", "Composite QoS index", "fig_results_09_qos_vs_channel.png")
sweep_plot("channel_sweep", "noise_offset_db", "energy_per_node_round_mJ", "Noise floor offset (dB)", "Energy per node per round (mJ)", "fig_results_10_energy_vs_channel.png")

# Fig. 11 and 12: GA shift and lifetime bars
for sec, metric, ylab, fname in (("ga_shift", "cross_latency_s", "Threshold-crossing latency (s)", "fig_results_11_ga_shift_latency.png"),
                                 ("ga_shift", "energy_per_node_round_mJ", "Energy per node per round (mJ)", "fig_results_12_ga_shift_energy.png")):
    fig, ax = plt.subplots(figsize=(W, H))
    bars(ax, METHODS, [RESULTS[sec][n][metric]["mean"] for n in METHODS], [RESULTS[sec][n][metric]["std"] for n in METHODS], [COL[n] for n in METHODS])
    ax.set_ylabel(ylab)
    ax.grid(axis="y", alpha=0.25, lw=0.4)
    finish(fig, ax, fname, legend=False)
fig, ax = plt.subplots(figsize=(W, H))
bars(ax, METHODS, [RESULTS["lifetime"][n]["mean"] for n in METHODS], [RESULTS["lifetime"][n]["std"] for n in METHODS], [COL[n] for n in METHODS])
ax.set_ylabel("Rounds to first node depletion")
ax.grid(axis="y", alpha=0.25, lw=0.4)
finish(fig, ax, "fig_results_13_network_lifetime.png", legend=False)

# Fig. 14: latency CDF
fig, ax = plt.subplots(figsize=(W, H))
for n in METHODS:
    x = np.sort(RESULTS["latency_samples"][n])
    ax.plot(x, np.arange(1, len(x) + 1) / len(x), color=COL[n], label=n)
ax.axvline(DEADLINE_SEC, color="gray", ls=":", lw=0.8)
ax.set_xlabel("Task completion latency (s)")
ax.set_ylabel("Empirical CDF")
ax.grid(alpha=0.25, lw=0.4)
finish(fig, ax, "fig_results_14_latency_cdf.png")

# Fig. 15: energy latency trade off
fig, ax = plt.subplots(figsize=(W, H))
for n in METHODS:
    d = RESULTS["default"][n]
    ax.errorbar(d["energy_per_node_round_mJ"]["mean"], d["cross_latency_s"]["mean"], xerr=d["energy_per_node_round_mJ"]["std"],
                yerr=d["cross_latency_s"]["std"], fmt=MRK[n], color=COL[n], capsize=2, label=n, elinewidth=0.6)
ax.set_xlabel("Energy per node per round (mJ)")
ax.set_ylabel("Threshold-crossing latency (s)")
ax.grid(alpha=0.25, lw=0.4)
finish(fig, ax, "fig_results_15_energy_latency.png")

# Fig. 16: ablation
ab = ["InterventionROP-Edge"] + ABLATIONS
fig, axs = plt.subplots(1, 2, figsize=(W, H + 0.2))
for ax, metric, ylab in zip(axs, ("cross_latency_s", "energy_per_node_round_mJ"), ("Crossing latency (s)", "Energy (mJ)")):
    ax.bar(np.arange(len(ab)), [RESULTS["default"][n][metric]["mean"] for n in ab], yerr=[RESULTS["default"][n][metric]["std"] for n in ab],
           color=["#1f4e9c"] + ["#9db7e0"] * len(ABLATIONS), width=0.65, capsize=2, error_kw={"elinewidth": 0.6})
    ax.set_xticks(np.arange(len(ab)))
    ax.set_xticklabels(["Full", "No do", "No twin", "No surr.", "Corr. u", "Unstrat. u"], rotation=45, ha="right")
    ax.set_ylabel(ylab)
    ax.grid(axis="y", alpha=0.25, lw=0.4)
finish(fig, axs[0], "fig_results_16_ablation.png", legend=False)
print(sorted(p.name for p in FIG_DIR.glob("*.png")))

## 12. Save and download

In [ ]:
json.dump(RESULTS, open(OUT_DIR / "results.json", "w"), indent=1)
archive = shutil.make_archive(str(OUT_DIR.parent / PAPER), "zip", OUT_DIR)
print("Saved:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)